In [101]:
import json
import requests
from PIL import Image
from pathlib import Path
import pandas as pd
import time

In [102]:
def open_json_file(json_file_name):
    '''
    The function opens a JSON file and returns it as a dictionary. It is used to process the iiif-manifest.
    '''
    with open(json_file_name, 'r') as json_file:
        return json.load(json_file)

In [103]:
def images_data_iiif_2(manifest: dict, library_domain: str) -> tuple:
    """
    Extracts image information from an IIIF v2 manifest.

    Args:
        manifest (dict): The JSON data of the IIIF manifest.
        library_domain (str): The domain of the library hosting the manifest.

    Returns:
        tuple: A tuple containing:
            - ms_name (str): The manuscript's name extracted from the manifest.
            - images_info (list of dict): A list containing image metadata, where each
              dictionary includes:
                - canvasNum (int): The index of the canvas.
                - imageLabel (str): The label of the image.
                - urlImage (str): The URL of the image.
                - imageFormat (str): The format of the image (e.g., jpg, png).
                - imageWidthAsDeclared (int): The declared width of the image.
                - imageHeightAsDeclared (int): The declared height of the image.

    Raises:
        Exception: If an error occurs during processing, an error message is printed,
                  and (None, []) is returned.
    """
    

    if not manifest:
            print("Failed to read the manifest.")
            raise ValueError("Empty manifest")
    
    try:
        ms_name = manifest.get('label', 'Unknown manuscript')
        canvases = manifest['sequences'][0]['canvases']
        images_info = []

        for idx, canvas in enumerate(canvases):
            image_label = canvas.get('label', 'Unknown label')

            image_resource = canvas['images'][0]['resource']
            image_url = image_resource['@id']
            image_width_declared = image_resource.get('width', 0)
            image_height_declared = image_resource.get('height', 0)

            if library_domain == 'digi.vatlib.it':
                # For Vatican, build a clean IIIF URL from the base image URL
                base_url = image_resource['service']['@id']
                image_url = base_url + '/full/full/0/default.jpg'

            image_format = image_url.split('.')[-1]

            images_info.append({
                'canvasNum': idx,
                'imageLabel': image_label,
                'urlImage': image_url,
                'imageFormat': image_format,
                'imageWidthAsDeclared': image_width_declared,
                'imageHeightAsDeclared': image_height_declared
            })

        print(f"Found {len(images_info)} images in the manifest.")
        return ms_name, images_info

    except Exception as e:
        print(f"An error occurred: {e}")
        return None, []

In [104]:
def images_data_iiif_3(manifest: dict, library_domain: str) -> tuple:
    """
    Extracts image information from an IIIF v3 manifest.

    Args:
        manifest (dict): The JSON data of the IIIF manifest.
        library_domain (str): The domain of the library hosting the manifest.

    Returns:
        tuple: A tuple containing:
            - ms_name (str): The manuscript's name extracted from the manifest.
            - images_info (list of dict): A list containing image metadata such as:
                - canvasNum (int): The index of the canvas.
                - imageLabel (str): The label of the image.
                - urlImage (str): The download URL of the image.
                - imageFormat (str): The format of the image (e.g., jpg, png).
                - imageWidthAsDeclared (int): The declared width of the image.
                - imageHeightAsDeclared (int): The declared height of the image.

    Raises:
        Exception: If an error occurs during processing, an error message is printed,
                  and (None, []) is returned.
    """
    
    if not manifest:
            print("Failed to read the manifest.")
            raise ValueError("Empty manifest")
    try:
    
        ms_name = manifest['label']['en'][0]
        canvases = manifest['items']
        images_info = []

        for idx, canvas in enumerate(canvases):
            image_label = (
                canvas['label']['en'][0]
                if 'label' in canvas and 'en' in canvas['label']
                else "Unknown label"
            )
            image_width_declared = canvas.get('width', 0)
            image_height_declared = canvas.get('height', 0)

            image_url = canvas['items'][0]['items'][0]['body']['id']

            if library_domain == 'digi.vatlib.it':
                download_url = image_url + '/full/max/0/default.jpg'
            else:
                # Replace the size segment with 'max' to retrieve the largest authorised size.
                download_url = image_url.replace(image_url.split('/')[-3], 'max')

            image_format = download_url.split('.')[-1]

            images_info.append({
                'canvasNum': idx,
                'imageLabel': image_label,
                'urlImage': download_url,
                'imageFormat': image_format,
                'imageWidthAsDeclared': image_width_declared,
                'imageHeightAsDeclared': image_height_declared
            })

        print(f"Found {len(images_info)} images in the manifest.")
        return ms_name, images_info

    except Exception as e:
        print(f"An error occurred: {e}")
        return None, []

In [ ]:
def download_from_iiif(manifest, iiif_version, library_domain, output_folder: Path, requests_pause:int) -> tuple:
    """
    Downloads all images from an IIIF manifest into a dedicated folder.

    Args:
        iiif_url (str): The URL of the IIIF manifest.
        output_folder (str): The root directory where images will be saved.

    Returns:
        tuple: A tuple containing:
            - ms_name (str): The manuscript name used as the folder name.
            - images_info (list of dict): The list of image metadata, updated with
              download results (imageFileName, folderPath, htmlCode).
    """
    if manifest is None:
        return None, []

    if iiif_version == 3:
        ms_name, images_info = images_data_iiif_3(manifest, library_domain)
    else:
        ms_name, images_info = images_data_iiif_2(manifest, library_domain)

    if not ms_name or not images_info:
        return ms_name, []
    
    output_folder.mkdir(parents=True, exist_ok=True)

    for idx, image in enumerate(images_info):
        url = image['urlImage']
        image_filename = f"image_{idx + 1}.{image['imageFormat']}"
        image_path = output_folder / image_filename

        image['imageFileName'] = image_filename
        image['folderPath'] = output_folder
        image['httpCode'] = ''
        image['imageWidthAsDownloaded'] = ''
        image['imageHeightAsDownloaded'] = ''

        if image_path.exists():
            print(f"Image {image_path} already dowloaded ")
            http_code = 200
            with Image.open(image_path) as img:
                imgWidthAsDownloaded, imgHeightAsDownloaded = img.size
        else:

            try:
                response = requests.get(url)
                http_code = response.status_code
                print(f"Downloading {image_filename} — HTTP status: {http_code}")

                if http_code == 200:
                    with open(image_path, 'wb') as f:
                        f.write(response.content)
                    image['httpCode'] = http_code
                    
                    with Image.open(image_path) as img:
                        imgWidthAsDownloaded, imgHeightAsDownloaded = img.size
                    image['imageWidthAsDownloaded'] = imgWidthAsDownloaded
                    image['imageHeightAsDownloaded'] = imgHeightAsDownloaded
                
                elif http_code == 429:
                    print(f"Error {http_code}. Rate limit exceeded. Pausing for {requests_pause} seconds.")
                    time.sleep(requests_pause)
        
                else:
                    print(f"Download error for {image_filename}: {http_code}")
                    image['httpCode'] = http_code

            except Exception as e:
                print(f"Exception while downloading {image_filename}: {e}")
                image['httpCode'] = 'Exception'
                image['imageWidthAsDownloaded'] = ''
                image['imageHeightAsDownloaded'] = ''

    df = pd.DataFrame(images_info)
    csv_path = output_folder / f"{ms_name}_image_data.csv"
    if csv_path.exists():
        df = df.set_index('canvasNum')
        df_existing = pd.read_csv(csv_path).set_index('canvasNum')
        df_existing.update(df)
        df_existing.reset_index().to_csv(csv_path, index=False)
        print(f"Image data updated to {csv_path}")
    else:
        df.to_csv(csv_path, index=False)
        print(f"Image data saved to {csv_path}")

In [ ]:
def download_data(manifests_csv:str, data_folder:str, requests_pause:int):
    """
    This function launches the download of all manifests in the list of manifest given as a CSV file (parameter 'list_of_manifests'). For each manifest, a folder will be created containing a copy 
    of the iiif-manifest file, the images and a CSV file documenting the downloads.
    """
    df = pd.read_csv(manifests_csv, sep=';')

    # Browse each line of the CSV file
    for i, row in df.iterrows():
        manifest_url = str(row['Manifest_URL'])
        ms_name = str(row['MS_name'])

        output_folder = Path(data_folder) / ms_name
        output_folder.mkdir(parents=True, exist_ok=True)

        try:
            response = requests.get(manifest_url)

            if response.status_code == 200:
                manifest = response.json()
                print(f"Manifest {manifest_url} loaded with success.")
                
                ms_path = Path(output_folder) / (ms_name + '.json')
                with open(ms_path, 'w', encoding='utf-8') as iiif:
                    json.dump(manifest, iiif, ensure_ascii=False, indent=2)
                
                print(f"Manifest saved in {ms_path}.")

                # Extract the IIIF version from the '@context' field.
                # '@context' can be a string or a list (IIIF v3 sometimes uses a list).
                context = manifest['@context']
                if isinstance(context, list):
                    context_str = next((c for c in context if 'iiif.io' in c), context[-1])
                else:
                    context_str = context
                iiif_version = int(context_str.split('/')[-2])

                # Extract the library domain from the URL
                library_domain = manifest_url.split('/')[2]
                # return manifest, iiif_version, library_domain

            else:
                print(f"Error loading the manifest: {response.status_code}")
                continue
            
            download_from_iiif(manifest, iiif_version, library_domain, output_folder, requests_pause)

        except Exception as e:
            print(f"An error occurred while loading or saving the manifest: {e}")
            raise
        


In [107]:
manifests_csv = '/Users/marioncharpier/Documents/GitHub/VARIA/manifests_list.csv'
data_folder = '/Users/marioncharpier/Documents/GitHub/VARIA/Test'
requests_pause = 5
download_data(manifests_csv, data_folder, requests_pause)

Manifest https://bl.digirati.io/iiif/ark:/81055/vdc_100055965341.0x000001 loaded with success.
Manifest saved in /Users/marioncharpier/Documents/GitHub/VARIA/Test/Add_11283/Add_11283.json.
Found 98 images in the manifest.
Image /Users/marioncharpier/Documents/GitHub/VARIA/Test/Add_11283/image_1.jpg already dowloaded 
Image data saved to /Users/marioncharpier/Documents/GitHub/VARIA/Test/Add_11283/add ms 11283_image_data.csv
Manifest https://digi.vatlib.it/iiif/MSS_Reg.lat.258/manifest.json loaded with success.
Manifest saved in /Users/marioncharpier/Documents/GitHub/VARIA/Test/MSS_Reg.lat.258/MSS_Reg.lat.258.json.
Found 114 images in the manifest.
Image data saved to /Users/marioncharpier/Documents/GitHub/VARIA/Test/MSS_Reg.lat.258/Reg.lat.258_image_data.csv
